# 10 — Focused artifact inventory

This notebook inventories outputs; it does not repeat stale headline claims or run inferential tests on diagnostic N=3 data. Missing artifacts remain PENDING, never zero. Each owning notebook supplies N and units.


In [ ]:
import os
from pathlib import Path
import pandas as pd
from wafer_analysis.focused import FOLLOWUP_ARTIFACTS, artifact_inventory, passed_artifacts
from wafer_analysis.paths import find_canonical_ledger, resolve_result_batch

root=os.environ.get('WAFER_SUMMARY_DIR')
batch_id=os.environ.get('WAFER_EVAL_BATCH_ID')
available=set()
question_experiment={
 'eKuiper latency tail':'e-perf-10','target load versus saturation':'e-perf-10',
 'branch-A throughput and latency':'e-iso-7','epoch recovery':'e-iso-4',
 'startup cache state':'e-perf-9','bounded queue pressure':'e-backpressure',
 'internal and sink-observed hot-swap timing':'e-swap-1',
}
for question,_,artifact in FOLLOWUP_ARTIFACTS:
    experiment=question_experiment[question]
    diagnostic=str(Path(root)/experiment) if root and (Path(root)/experiment).is_dir() else None
    try:
        batch=resolve_result_batch(experiment,diagnostic_path=diagnostic,batch_id=batch_id)
    except (FileNotFoundError,RuntimeError,ValueError):
        continue
    if passed_artifacts(batch, artifact):
        available.add(artifact)
if batch_id:
    try:
        ledger=find_canonical_ledger(batch_id)
        if (ledger/'rate-sweep-summary.json').is_file():
            available.add('rate-sweep-summary.json')
    except (FileNotFoundError,ValueError):
        pass
inventory=artifact_inventory(available)
display(inventory)
print('N and units are reported by each owning notebook; uncertainty is descriptive and thesis_evidence=false for the focused pilot.')
print('Power label where applicable: Raspberry Pi 5 PMIC internal-rail proxy; not total board power.')
